# Setup and Installation
Install and import all required libraries including PyTorch, Transformers, Sentence Transformers, NLTK, pandas, NumPy, and scikit-learn. Set up environment configurations and global parameters.

In [ ]:
# Install required libraries
!pip install torch transformers sentence-transformers nltk pandas numpy scikit-learn graphviz

# Deep learning libraries
import torch
from transformers import BertModel, BertTokenizer
from sentence_transformers import SentenceTransformer

# NLP libraries
import nltk

# Data processing libraries
import pandas as pd
import numpy as np

# Machine learning libraries
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score

# Download necessary NLTK datasets and models
nltk.download('treebank')      # Penn Treebank sample corpus
nltk.download('punkt')         # Pre-trained tokenizer
nltk.download('punkt_tab')     # Tab-delimited version of punkt
nltk.download('averaged_perceptron_tagger')  # ML-based POS tagger model
from nltk.corpus import treebank

import os

#-------------------------------------------------------------------------------
# Set up the appropriate device (CUDA, MPS, or CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# Define model names
BERT_MODEL_NAME = 'bert-base-uncased'  # Standard BERT model (not case-sensitive)
SENTENCE_TRANSFORMER_MODEL_NAME = 'all-MiniLM-L6-v2'  # Lightweight sentence embedding model

# Load transformer models
print(f"Loading {BERT_MODEL_NAME}...")
bert_model = BertModel.from_pretrained(BERT_MODEL_NAME).to(device)
bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

print(f"Loading {SENTENCE_TRANSFORMER_MODEL_NAME}...")
sentence_transformer_model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL_NAME).to(device)

print("loaded")

# Pipeline Architecture
Define the core pipeline architecture with modular components. Create base classes that establish interfaces for each module. Visualize the pipeline structure using diagrams to illustrate data flow.

In [ ]:
# Define the core pipeline architecture with modular components

# Base class for pipeline modules
class PipelineModule:
    def __init__(self, name):
        self.name = name

    def process(self, data):
        raise NotImplementedError("Each module must implement the process method.")

# Treebank data loading module
class TreebankLoaderModule(PipelineModule):
    def __init__(self, name):
        super().__init__(name)
    
    def process(self, data=None):
        from nltk.corpus import treebank
        sentences = treebank.tagged_sents()
        
        data = []
        for sent in sentences:
            words, tags = zip(*sent)
            sentence = ' '.join(words)
            for word, tag in sent:
                data.append({'sentence': sentence, 'word': word, 'tag': tag})
        
        return pd.DataFrame(data)

# Preprocessing module
class PreprocessingModule(PipelineModule):
    def __init__(self, name):
        super().__init__(name)

    def process(self, data):
        # Tokenize sentences using NLTK
        data['tokens'] = data['sentence'].apply(nltk.word_tokenize)
        return data

# Embedding creation module
class EmbeddingModule(PipelineModule):
    def __init__(self, name, model, tokenizer):
        super().__init__(name)
        self.model = model
        self.tokenizer = tokenizer

    def process(self, data):
        # Generate embeddings using BERT
        def get_embeddings(sentence):
            inputs = self.tokenizer(sentence, return_tensors='pt', truncation=True, padding=True).to(device)
            outputs = self.model(**inputs)
            return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().detach().numpy()

        data['embeddings'] = data['sentence'].apply(get_embeddings)
        return data

# Model training module
class ModelTrainingModule(PipelineModule):
    def __init__(self, name, model):
        super().__init__(name)
        self.model = model

    def process(self, data):
        # Train logistic regression model
        X = np.vstack(data['embeddings'].values)
        y = data['label']
        self.model.fit(X, y)
        return self.model

# Evaluation module
class EvaluationModule(PipelineModule):
    def __init__(self, name, model):
        super().__init__(name)
        self.model = model

    def process(self, data):
        # Evaluate model performance
        X = np.vstack(data['embeddings'].values)
        y = data['label']
        y_pred = self.model.predict(X)
        accuracy = accuracy_score(y, y_pred)
        precision = precision_score(y, y_pred, average='weighted')
        recall = recall_score(y, y_pred, average='weighted')
        f1 = f1_score(y, y_pred, average='weighted')
        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Visualizing the pipeline structure using diagrams to illustrate data flow
from graphviz import Digraph

def visualize_pipeline():
    dot = Digraph(comment='Pipeline Architecture')
    dot.node('A', 'TreebankLoaderModule')
    dot.node('B', 'PreprocessingModule')
    dot.node('C', 'EmbeddingModule')
    dot.node('D', 'ModelTrainingModule')
    dot.node('E', 'EvaluationModule')

    dot.edges(['AB', 'BC', 'CD', 'DE'])
    return dot

# Example usage
# data_loader = DataLoaderModule('DataLoader', 'path/to/data.csv')
preprocessing = PreprocessingModule('Preprocessing')
embedding = EmbeddingModule('Embedding', bert_model, bert_tokenizer)
model_training = ModelTrainingModule('ModelTraining', LogisticRegression())
evaluation = EvaluationModule('Evaluation', model_training.model)

# Visualize the pipeline
visualize_pipeline()

# Data Loading Module
Implement flexible data loaders for different datasets (Treebank, custom datasets). Create abstract interfaces for consistent data loading patterns. Include examples using built-in datasets and custom data formats.

In [ ]:
# Data Loading Module

class TreebankLoaderModule(PipelineModule):
    def __init__(self, name):
        super().__init__(name)
    
    def process(self, data=None):
        # Load directly from NLTK - no file path needed
        from nltk.corpus import treebank
        sentences = treebank.tagged_sents()
        
        data = []
        for sent in sentences:
            words, tags = zip(*sent)
            sentence = ' '.join(words)
            for word, tag in sent:
                data.append({'sentence': sentence, 'word': word, 'tag': tag})
        
        return pd.DataFrame(data)

# Create module instance
treebank_module = TreebankLoaderModule("treebank_loader")

# Load data using the pipeline module
treebank_data = treebank_module.process()

# Display loaded data
print(treebank_data.head())

# Preprocessing Module
Develop text preprocessing utilities including tokenization, normalization, and filtering. Create pipeline-compatible transformer classes for text preprocessing. Implement feature extraction methods specific to linguistic properties.

In [ ]:
# Preprocessing Module

# Import necessary libraries for preprocessing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.base import TransformerMixin

# Download NLTK stopwords
nltk.download('stopwords')

# Define text preprocessing utilities
class TextPreprocessor(TransformerMixin):
    def __init__(self, remove_stopwords=True, lowercase=True):
        self.remove_stopwords = remove_stopwords
        self.lowercase = lowercase
        self.stop_words = set(stopwords.words('english')) if remove_stopwords else None

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return X.apply(self._preprocess_text)

    def _preprocess_text(self, text):
        tokens = word_tokenize(text)
        if self.lowercase:
            tokens = [token.lower() for token in tokens]

        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in self.stop_words]
        return tokens

# pipeline-compatible transformer class for text preprocessing
class PreprocessingModule(PipelineModule):
    def __init__(self, name, remove_stopwords=True, lowercase=True):
        super().__init__(name)
        self.preprocessor = TextPreprocessor(remove_stopwords, lowercase)

    def process(self, data):
        # Apply text preprocessing
        data['processed_text'] = self.preprocessor.transform(data['sentence'])
        return data

# Example usage of the PreprocessingModule
preprocessing = PreprocessingModule('Preprocessing')
processed_data = preprocessing.process(treebank_data)

# Display processed data
print(processed_data.head())

# Embedding Generation Module
Create wrappers for various LLMs (BERT, GPT, etc.). Implement layer-specific embedding extraction. Add caching mechanisms for efficiency. Include visualization tools for embedding analysis.

In [ ]:
# Embedding Generation Module

import os
import pickle
from transformers import GPT2Model, GPT2Tokenizer
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

class EmbeddingModule(PipelineModule):
    def __init__(self, name, model, tokenizer, cache_dir='cache'):
        super().__init__(name)
        self.model = model
        self.tokenizer = tokenizer
        self.cache_dir = cache_dir
        if not os.path.exists(cache_dir):
            os.makedirs(cache_dir)

    def process(self, data):
        # Generate embeddings using the specified model
        def get_embeddings(sentence):
            cache_path = os.path.join(self.cache_dir, f"{hash(sentence)}.pkl")
            if os.path.exists(cache_path):
                with open(cache_path, 'rb') as f:
                    embeddings = pickle.load(f)
            else:
                inputs = self.tokenizer(sentence, return_tensors='pt', truncation=True, padding=True).to(device)
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().detach().numpy()
                with open(cache_path, 'wb') as f:
                    pickle.dump(embeddings, f)
            return embeddings

        data['embeddings'] = data['sentence'].apply(get_embeddings)
        return data

    def visualize_embeddings(self, data, n_components=2):
        embeddings = np.vstack(data['embeddings'].values)
        pca = PCA(n_components=n_components)
        reduced_embeddings = pca.fit_transform(embeddings)
        
        from sklearn.preprocessing import LabelEncoder
        le = LabelEncoder()
        numeric_tags = le.fit_transform(data['tag'])
        
        plt.figure(figsize=(10, 7))
        scatter = plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], 
                            c=numeric_tags, cmap='viridis')
        
        unique_tags = le.classes_
        handles = [plt.Line2D([0], [0], marker='o', color='w', 
                            markerfacecolor=plt.cm.viridis(i/len(unique_tags)), 
                            label=tag, markersize=8) 
                for i, tag in enumerate(unique_tags[:10])]  # Show only first 10 tags
        
        plt.legend(handles=handles, title='POS Tags', loc='upper right')
        plt.colorbar()
        plt.title('PCA of Embeddings')
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.show()

# Example usage of the EmbeddingModule
# embedding = EmbeddingModule('Embedding', GPT2Model, GPT2Tokenizer)
embedding = EmbeddingModule('Embedding', bert_model, bert_tokenizer)
embedded_data = embedding.process(treebank_data)

# Visualize the embeddings
embedding.visualize_embeddings(embedded_data)

# Probing Task Configuration
Define a framework for configuring different probing tasks. Create task-specific data processing pipelines. Implement sample probing tasks (e.g., POS tagging, syntactic structure detection).

In [ ]:
# Probing Task Configuration

# Define a framework for configuring different probing tasks
class ProbingTask:
    def __init__(self, name, data_loader, preprocessing, embedding, model_training, evaluation):
        self.name = name
        self.data_loader = data_loader
        self.preprocessing = preprocessing
        self.embedding = embedding
        self.model_training = model_training
        self.evaluation = evaluation

    def run(self):
        # Load data
        data = self.data_loader.process()
        # Preprocess data
        data = self.preprocessing.process(data)
        # Generate embeddings
        data = self.embedding.process(data)
        # Train model
        model = self.model_training.process(data)
        # Evaluate model
        results = self.evaluation.process(data)
        return results

# Create task-specific data processing pipelines
class POSTaggingTask(ProbingTask):
    def __init__(self, data_loader, preprocessing, embedding, model_training, evaluation):
        super().__init__('POS Tagging', data_loader, preprocessing, embedding, model_training, evaluation)

class SyntacticStructureTask(ProbingTask):
    def __init__(self, data_loader, preprocessing, embedding, model_training, evaluation):
        super().__init__('Syntactic Structure Detection', data_loader, preprocessing, embedding, model_training, evaluation)

# Implement sample probing tasks (e.g., POS tagging, syntactic structure detection)
# Example usage of POS Tagging Task
pos_tagging_task = POSTaggingTask(data_loader, preprocessing, embedding, model_training, evaluation)
pos_tagging_results = pos_tagging_task.run()
print(f"POS Tagging Task Results: {pos_tagging_results}")

# Example usage of Syntactic Structure Detection Task
syntactic_structure_task = SyntacticStructureTask(data_loader, preprocessing, embedding, model_training, evaluation)
syntactic_structure_results = syntactic_structure_task.run()
print(f"Syntactic Structure Detection Task Results: {syntactic_structure_results}")

# Model Training Module
Implement probing classifiers using scikit-learn and PyTorch. Add hyperparameter optimization utilities. Create training loops with appropriate callbacks. Include cross-validation implementations.

In [ ]:
# Model Training Module

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer

class ModelTrainingModule(PipelineModule):
    def __init__(self, name, model, param_grid=None, cv=5):
        super().__init__(name)
        self.model = model
        self.param_grid = param_grid
        self.cv = cv
        self.best_model = None

    def process(self, data):
        X = np.vstack(data['embeddings'].values)
        y = data['label']

        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        if self.param_grid:
            # Perform hyperparameter optimization using GridSearchCV
            grid_search = GridSearchCV(self.model, self.param_grid, cv=self.cv, scoring=make_scorer(f1_score, average='weighted'))
            grid_search.fit(X_scaled, y)
            self.best_model = grid_search.best_estimator_
        else:
            # Train the model without hyperparameter optimization
            self.model.fit(X_scaled, y)
            self.best_model = self.model

        return self.best_model

    def cross_validate(self, data):
        X = np.vstack(data['embeddings'].values)
        y = data['label']

        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Perform cross-validation
        scores = cross_val_score(self.best_model, X_scaled, y, cv=self.cv, scoring=make_scorer(f1_score, average='weighted'))
        return scores

# Example usage of the ModelTrainingModule with hyperparameter optimization
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l2']
}
model_training = ModelTrainingModule('ModelTraining', LogisticRegression(), param_grid=param_grid)
trained_model = model_training.process(embedded_data)
cross_val_scores = model_training.cross_validate(embedded_data)

print(f"Cross-validation scores: {cross_val_scores}")
print(f"Mean cross-validation score: {np.mean(cross_val_scores)}")

# Evaluation Module
Develop comprehensive evaluation metrics (accuracy, F1, etc.). Implement statistical significance testing. Create visualizations for interpreting results. Add comparison utilities against baselines.

In [ ]:
# Evaluation Module

import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

class EvaluationModule(PipelineModule):
    def __init__(self, name, model):
        super().__init__(name)
        self.model = model

    def process(self, data):
        # Evaluate model performance
        X = np.vstack(data['embeddings'].values)
        y = data['label']
        y_pred = self.model.predict(X)
        accuracy = accuracy_score(y, y_pred)
        precision = precision_score(y, y_pred, average='weighted')
        recall = recall_score(y, y_pred, average='weighted')
        f1 = f1_score(y, y_pred, average='weighted')
        return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

    def statistical_significance_test(self, data, baseline_model):
        # Perform statistical significance testing
        X = np.vstack(data['embeddings'].values)
        y = data['label']
        y_pred = self.model.predict(X)
        y_baseline_pred = baseline_model.predict(X)
        
        # Calculate F1 scores for both models
        f1_model = f1_score(y, y_pred, average='weighted')
        f1_baseline = f1_score(y, y_baseline_pred, average='weighted')
        
        # Perform t-test
        t_stat, p_value = ttest_ind([f1_model], [f1_baseline])
        return {'t_stat': t_stat, 'p_value': p_value}

    def plot_metrics(self, metrics):
        # Plot evaluation metrics
        labels = list(metrics.keys())
        values = list(metrics.values())
        
        plt.figure(figsize=(10, 5))
        plt.bar(labels, values, color='skyblue')
        plt.xlabel('Metrics')
        plt.ylabel('Scores')
        plt.title('Evaluation Metrics')
        plt.ylim(0, 1)
        plt.show()

    def compare_with_baseline(self, data, baseline_model):
        # Compare model performance with baseline
        baseline_metrics = self.process(data)
        current_metrics = self.process(data)
        
        comparison = {metric: current_metrics[metric] - baseline_metrics[metric] for metric in current_metrics}
        return comparison

# Example usage of the EvaluationModule
evaluation = EvaluationModule('Evaluation', trained_model)
evaluation_results = evaluation.process(embedded_data)
print(f"Evaluation Results: {evaluation_results}")

# Perform statistical significance testing
baseline_model = LogisticRegression()  # Assuming a baseline model is defined and trained
baseline_model.fit(np.vstack(embedded_data['embeddings'].values), embedded_data['label'])
significance_test_results = evaluation.statistical_significance_test(embedded_data, baseline_model)
print(f"Statistical Significance Test Results: {significance_test_results}")

# Plot evaluation metrics
evaluation.plot_metrics(evaluation_results)

# Compare with baseline
comparison_results = evaluation.compare_with_baseline(embedded_data, baseline_model)
print(f"Comparison with Baseline: {comparison_results}")

# Example Probing Tasks
Implement the verb classification task mentioned in the proposal. Add syntactic structure probing based on Shi et al. (2016). Include semantic information probing inspired by Tenney et al. (2019).

In [ ]:
# Example Probing Tasks

# Verb Classification Task
class VerbClassificationTask(ProbingTask):
    def __init__(self, data_loader, preprocessing, embedding, model_training, evaluation):
        super().__init__('Verb Classification', data_loader, preprocessing, embedding, model_training, evaluation)

# Syntactic Structure Probing Task
class SyntacticStructureProbingTask(ProbingTask):
    def __init__(self, data_loader, preprocessing, embedding, model_training, evaluation):
        super().__init__('Syntactic Structure Probing', data_loader, preprocessing, embedding, model_training, evaluation)

# Semantic Information Probing Task
class SemanticInformationProbingTask(ProbingTask):
    def __init__(self, data_loader, preprocessing, embedding, model_training, evaluation):
        super().__init__('Semantic Information Probing', data_loader, preprocessing, embedding, model_training, evaluation)

# Example usage of Verb Classification Task
verb_classification_task = VerbClassificationTask(data_loader, preprocessing, embedding, model_training, evaluation)
verb_classification_results = verb_classification_task.run()
print(f"Verb Classification Task Results: {verb_classification_results}")

# Example usage of Syntactic Structure Probing Task
syntactic_structure_probing_task = SyntacticStructureProbingTask(data_loader, preprocessing, embedding, model_training, evaluation)
syntactic_structure_probing_results = syntactic_structure_probing_task.run()
print(f"Syntactic Structure Probing Task Results: {syntactic_structure_probing_results}")

# Example usage of Semantic Information Probing Task
semantic_information_probing_task = SemanticInformationProbingTask(data_loader, preprocessing, embedding, model_training, evaluation)
semantic_information_probing_results = semantic_information_probing_task.run()
print(f"Semantic Information Probing Task Results: {semantic_information_probing_results}")

# Pipeline Demo: End-to-End
Create a complete demonstration of the pipeline using a sample probing task. Show how to customize each component. Include annotation explaining each step in detail.

In [ ]:
# Pipeline Demo: End-to-End

# Define a sample dataset for demonstration
sample_data = pd.DataFrame({
    'sentence': [
        'The quick brown fox jumps over the lazy dog.',
        'She sells sea shells by the sea shore.',
        'How much wood would a woodchuck chuck if a woodchuck could chuck wood?',
        'To be or not to be, that is the question.',
        'All that glitters is not gold.'
    ],
    'label': [1, 0, 1, 0, 1]  # Sample labels for demonstration purposes
})

# Save the sample dataset to a CSV file
sample_data.to_csv('sample_data.csv', index=False)

# Create instances of the pipeline modules
data_loader = DataLoaderModule('DataLoader', 'sample_data.csv')
preprocessing = PreprocessingModule('Preprocessing')
embedding = EmbeddingModule('Embedding', bert_model, bert_tokenizer)
model_training = ModelTrainingModule('ModelTraining', LogisticRegression())
evaluation = EvaluationModule('Evaluation', model_training.model)

# Define a complete pipeline for a sample probing task
class SampleProbingTask(ProbingTask):
    def __init__(self, data_loader, preprocessing, embedding, model_training, evaluation):
        super().__init__('Sample Probing Task', data_loader, preprocessing, embedding, model_training, evaluation)

# Run the sample probing task
sample_probing_task = SampleProbingTask(data_loader, preprocessing, embedding, model_training, evaluation)
sample_results = sample_probing_task.run()

# Print the results of the sample probing task
print(f"Sample Probing Task Results: {sample_results}")

# Annotate each step in detail
# Step 1: Load the data
data = data_loader.process()
print("Step 1: Loaded Data")
print(data.head())

# Step 2: Preprocess the data
data = preprocessing.process(data)
print("Step 2: Preprocessed Data")
print(data.head())

# Step 3: Generate embeddings
data = embedding.process(data)
print("Step 3: Generated Embeddings")
print(data.head())

# Step 4: Train the model
model = model_training.process(data)
print("Step 4: Trained Model")

# Step 5: Evaluate the model
results = evaluation.process(data)
print("Step 5: Evaluation Results")
print(results)

# Visualize the embeddings
embedding.visualize_embeddings(data)

# Plot evaluation metrics
evaluation.plot_metrics(results)